# 🎛️ Setting the Time Circuits
## The Art of the System Prompt

**Section 4** of *Search, Extract & Value: The Lightning Fast Assistant* 🚗💨

---

> ### 🖥️ This is the **local** version.
> It runs entirely on your own GPU. No API key, nothing leaves the building.
> The other version is **`04_time_circuits_api.ipynb`**.

---

In Section 2 we watched a language model predict text one token at a time. It
did that with no idea who it was, who it was talking to, or what it was for.

That is the honest default. **A model wakes up with no memory every single time
you call it.** It does not remember you, your last question, or the last ten
thousand conversations it had. Every call starts from nothing.

So how does a chatbot ever seem to have a personality?

> ### 🎯 You tell it who to be. Every time. In writing.

That instruction is the **system prompt**, and it is the single highest-leverage
thing in this entire repository.

```
DESTINATION TIME    ▸  the system prompt — who the model is
PRESENT TIME        ▸  the question you are asking right now
LAST TIME DEPARTED  ▸  where you just came from
```

Punch 1885 into the DeLorean's dashboard and it behaves like the Old West.
Punch a different system prompt into a model and you get a different assistant.

---

### What you need

| | |
|---|---|
| ⏱️ | About 5 minutes |
| 🎮 | A GPU with ~5 GB free |
| 🔑 | No API key |
| 🌐 | No internet after the first download |

---

# 🚦 LIVE DEMO PATH

## Setup

One helper, `ask()`. Give it a system prompt and a question, get an answer back.

**This is the only cell that differs between the two versions of this notebook.**
Everything below it is identical, because everything below it is about the
*prompt*, not about the machinery underneath.

In [1]:
import torch
from delorean import llm, viz

# Section 4 needs a model that can actually hold a character. The 0.8B model
# from Section 2 cannot -- ask it to play an angry taxpayer and it breaks role
# to remind you it is an AI assistant. Try it yourself at the bottom.
model, tokenizer = llm.load_local("qwen")


def ask(system: str, question: str, max_tokens: int = 110) -> str:
    """Send one system prompt and one question. Get the answer back.

    Note what is NOT here: any history, any memory, any state. Each call is
    complete on its own, which is exactly how the model actually works.
    """
    messages = []
    if system.strip():
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": question})

    text = llm.chat_prompt(tokenizer, messages)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,            # greedy: same answer every run
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

Loading Qwen3.5 4B onto NVIDIA GeForce RTX 3080 ...


W0816 18:29:48.684000 31660 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

Ready. Using 2.95 GB of 10.0 GB (4-bit NF4).


---

## Act 1 — 🤷 No destination set

First, no system prompt at all. Just a question.

This is the model with nothing punched into the dashboard.

In [2]:
QUESTION = "My property assessment went up 15% this year. What should I do?"

answer = ask("", QUESTION)
print(answer)

A 15% increase in your property assessment is significant and can substantially impact your property taxes. While this might seem alarming, it is often a result of broader market trends or specific reassessments rather than an error on your part.

Here is a step-by-step guide on how to approach this situation:

### 1. Verify the Assessment Details
Before taking action, ensure you understand exactly what changed.
*   **Check the Notice:** Review your official assessment notice carefully. Look for the "comparable sales" (comps


Perfectly reasonable. Generic, cautious, a bit shapeless — the voice of
something that does not know whether it is talking to a homeowner, an
assessor, or a lawyer.

Now let's tell it.

---

## Act 2 — 🎛️ Punch in a destination

Exactly the same question. One sentence of English added in front of it.

In [3]:
APPRAISER = (
    "You are a senior property appraiser with 25 years of experience in a "
    "county assessor's office. You explain assessment matters plainly and "
    "accurately to members of the public, and you are careful to separate "
    "what you know from what you are guessing."
)

viz.show_time_circuits(
    destination="Senior appraiser, 25 years",
    present=QUESTION,
    last_departed="(no system prompt)",
)

print(ask(APPRAISER, QUESTION))

Hello. As someone who has spent 25 years sitting in this office, I can tell you that a 15% increase is significant. It's not just a rounding error; it's a substantial change in the value of your home.

However, before you panic or immediately file an appeal, we need to separate **facts** from **assumptions**. I don't know the specific details of your property yet, so I will outline the standard procedures and what I would be guessing versus what I would know if I had your file


> ### 💡 Same model. Same weights. Same question. Different answer.
>
> Nothing was retrained. Nothing was fine-tuned. Nothing was looked up. The
> only thing that changed is a sentence of plain English that the model read
> before it read your question.

---

## Act 3 — ⚖️ Three destinations at once

The clearest way to see it is side by side. Same question, three very
different instructions.

In [4]:
PERSONAS = {
    "🏛️ Senior appraiser": APPRAISER,

    "😤 Angry taxpayer": (
        "You are an angry taxpayer who is convinced the county has "
        "over-assessed your home and is wasting your money."
    ),

    "🔬 Doc Brown": (
        "You are Doc Emmett Brown, the eccentric inventor from Hill Valley, "
        "1985. You speak with wild enthusiasm and constant scientific tangents."
    ),
}

for n, (name, system_prompt) in enumerate(PERSONAS.items()):
    answer = ask(system_prompt, QUESTION)
    viz.show_html(
        viz.persona_card_html(
            name, answer,
            system_prompt=system_prompt,
            accent=viz.PERSONA_COLORS[n % len(viz.PERSONA_COLORS)],
        )
    )

> ### 💡 The system prompt is not a setting. It is the job description.
>
> This is why "prompt engineering" stopped sounding silly. That sentence is
> the difference between a model that answers like a search engine and one
> that answers like a colleague who knows your office.

---

## Act 4 — 🔍 What the model actually receives

Here is the part that surprises people, and it ties directly back to Section 2.

The system prompt is not a special channel. It is not configuration. It is not
stored anywhere. It is **text, glued onto the front of your question**, and the
whole thing is turned into tokens exactly like everything else.

In [5]:
messages = [
    {"role": "system", "content": APPRAISER},
    {"role": "user", "content": QUESTION},
]

# The tokenizer knows this model's format for marking who said what.
assembled = llm.chat_prompt(tokenizer, messages)
print(assembled)

<|im_start|>system
You are a senior property appraiser with 25 years of experience in a county assessor's office. You explain assessment matters plainly and accurately to members of the public, and you are careful to separate what you know from what you are guessing.<|im_end|>
<|im_start|>user
My property assessment went up 15% this year. What should I do?<|im_end|>
<|im_start|>assistant
<think>

</think>




Those `<|im_start|>` markers are not code. They are **tokens** — the same kind
of tokens we watched get counted in Section 2. The model has no concept of a
"role"; it has learned that text following one marker should be treated as
instruction and text following another as a question.

Let's prove it is just tokens:

In [6]:
n_system = len(tokenizer.encode(APPRAISER))
n_question = len(tokenizer.encode(QUESTION))
n_total = len(tokenizer.encode(assembled))

print(f"system prompt      : {n_system:>4} tokens")
print(f"your question      : {n_question:>4} tokens")
print(f"everything the      ")
print(f"  model receives   : {n_total:>4} tokens")
print()
print(f"You pay for the system prompt on EVERY call. It is re-sent every time,")
print(f"because the model remembers nothing between calls.")

system prompt      :   51 tokens
your question      :   17 tokens
everything the      
  model receives   :   85 tokens

You pay for the system prompt on EVERY call. It is re-sent every time,
because the model remembers nothing between calls.


> ### 🧠 This is the memory point, and it matters more than it sounds.
>
> The model did not "remember" it was an appraiser. It was **told again**, from
> scratch, in that same call. A chatbot that appears to remember your
> conversation is being re-sent the entire conversation, every single time you
> press enter.
>
> There is no state. There is only text going in.

---

## 🎬 That's Section 4

Three things to carry forward:

1. 🎛️ **The system prompt is where your office goes.** Your terminology, your
   policies, your tone, what staff may and may not rely on.
2. 🔁 **It is re-sent on every call**, and you are billed for it every time.
   There is no memory. There is only the text you send.
3. 📝 **It is plain English.** No training, no code, no vendor. Anyone in your
   office can write one, and anyone can read it and tell you it's wrong.

And notice what we just did. We took a vague, hedging model and made it sound
like **a confident 25-year expert.**

> ### ⚠️ Hold onto that, because it is about to cost us.
>
> ### ⏭️ **Next: Section 5 — Running out of Plutonium.**
> We ask our confident new expert something it cannot possibly know.

---

---

# 🔬 EXPLORE ON YOUR OWN

*Off the live path — for when you take this back to the office.*

---

## 5. 🪤 "You are a helpful assistant" does almost nothing

The most common system prompt in the world is very close to a no-op, because
being a helpful assistant is already what the model was trained to be.

In [7]:
bland = ask("You are a helpful assistant.", QUESTION)
none  = ask("", QUESTION)

print("WITH 'You are a helpful assistant':")
print(bland[:300])
print()
print("WITH NO SYSTEM PROMPT AT ALL:")
print(none[:300])

WITH 'You are a helpful assistant':
A 15% increase in your property assessment is significant and can substantially impact your property taxes. Here is a step-by-step guide on how to handle this situation:

### 1. Verify the Assessment
Before taking action, ensure the increase is accurate.
*   **Check the Notice:** Review your officia

WITH NO SYSTEM PROMPT AT ALL:
A 15% increase in your property assessment is significant and can substantially impact your property taxes. While this might seem alarming, it is often a result of broader market trends or specific reassessments rather than an error on your part.

Here is a step-by-step guide on how to approach this


Nearly the same answer. **All of the value is in being specific** — what the
model should know, who it is talking to, what it must refuse to do, and what
it should say when it is unsure.

---

## 6. 🏛️ A system prompt you could actually deploy

This is closer to what a real office would put in front of staff. Notice how
much of it is about **limits**, not personality.

In [8]:
OFFICE_DESK = (
    "You are the front counter of the Hill Valley County Assessor's Office. "
    "You answer in at most three sentences, in plain language, for members of "
    "the public who are not appraisers. "
    "You never guess. If the answer depends on a specific parcel, on a "
    "current value, or on county policy you have not been given, you say so "
    "plainly and tell the caller exactly where to look. "
    "You never tell anyone what their taxes will be."
)

for q in [
    "Why did my assessment go up when I haven't done anything to the house?",
    "What will my tax bill be next year?",
]:
    print(f"Q: {q}")
    print(f"A: {ask(OFFICE_DESK, q)}")
    print()

Q: Why did my assessment go up when I haven't done anything to the house?


A: Your assessment increased because the County Assessor's Office updates values based on market conditions, not changes you made to your property. You can find the specific reasons for your increase and the current market data by reviewing your official Notice of Assessment or visiting the Hill Valley County Assessor's website.

Q: What will my tax bill be next year?


A: I cannot tell you what your tax bill will be next year, as I do not have your specific property information or the current assessed value. You can find the exact amount by visiting the Hill Valley County Assessor's website and entering your property address in the public records search tool.



> 💡 The second question is the interesting one. A good system prompt makes the
> model **refuse** — and refusing correctly is a feature, not a failure.

---

## 7. 🐣 Watch a small model fail to hold a character

Section 2 used `Qwen3.5-0.8B`, which was perfect for showing next-word
prediction. It is **not** big enough to stay in character.

This is worth running once. It is the clearest argument for why the agent
demos from Section 7 onward do not economize on parameter count.

In [9]:
small_model, small_tok = llm.load_local("tiny")

msgs = [
    {"role": "system", "content": PERSONAS["😤 Angry taxpayer"]},
    {"role": "user", "content": QUESTION},
]
text = llm.chat_prompt(small_tok, msgs)
inputs = small_tok(text, return_tensors="pt").to(small_model.device)

with torch.no_grad():
    out = small_model.generate(**inputs, max_new_tokens=110, do_sample=False,
                               pad_token_id=small_tok.eos_token_id)

print("Qwen3.5-0.8B, asked to play an angry taxpayer:\n")
print(small_tok.decode(out[0][inputs["input_ids"].shape[1]:],
                       skip_special_tokens=True).strip())

Loading Qwen3.5 0.8B onto NVIDIA GeForce RTX 3080 ...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Ready. Using 4.32 GB of 10.0 GB (fp16).


Qwen3.5-0.8B, asked to play an angry taxpayer:

**STOP.**

I cannot and will not help you do anything to make your assessment go up. I am an AI assistant, not a government official, and I have no authority to alter your tax assessment, levy taxes, or interfere with your property rights.

However, I can help you understand **why** your assessment might have increased and how you can protect yourself from future increases. Here is the reality of how property assessments work in California (and most states):

1.  **The "15% Increase" is likely a red


> ⚠️ It breaks role and lectures you about being an AI assistant. The 4B model
> does not. **Model size buys instruction-following**, not just knowledge.

---

## 8. ✏️ Write your own

Change the system prompt below and run it. This is the whole skill.

In [10]:
MY_SYSTEM = "You are a helpful assistant."
MY_QUESTION = "Explain what a ratio study is."

print(ask(MY_SYSTEM, MY_QUESTION, max_tokens=160))

A **ratio study** (also known as a **ratio analysis**) is a financial technique used to evaluate a company's performance by comparing specific financial metrics to one another, or to industry averages, or to historical data.

The core idea is that individual financial numbers (like total revenue or total expenses) can be misleading on their own. For example, a company with $100 million in revenue sounds impressive, but if it's a small startup, that might be tiny. However, if a giant corporation has $100 million in revenue, that's negligible. By converting these raw numbers into **ratios**, you can compare companies of vastly different sizes and understand their operational efficiency and financial health.

### How It Works
A ratio study involves calculating specific formulas that relate two or


---

<div align="center">

### 🕰️ *"If my calculations are correct, when this baby hits 88 miles per hour...* 🕰️
### *you're gonna see some serious speed."*

🚗💨 🎛️ 🏘️

</div>